# Fine-tune a Remotion motion-design model with QLoRA

This notebook trains a small LoRA adapter using the public 12,081-example dataset. It starts with 1,000 training examples and 200 validation examples to keep a first Kaggle run manageable. Set the sample caps to `0` to use all 10,469 training rows and 837 validation rows. The test split is not loaded.

In Notebook settings, turn on **Internet** and a **GPU accelerator**. The example uses Qwen2.5-Coder-1.5B-Instruct with 4-bit QLoRA. GPU capacity and quotas vary; this is not a 31B full fine-tune.

In [ ]:
%pip install -q 'trl[peft]' bitsandbytes datasets accelerate
print('Install complete. If imports fail, restart the Kaggle session and rerun from the top.')

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

BASE = 'https://github.com/rohithcreater/remotion-ai-agents/raw/refs/heads/main'
WORK = Path('/kaggle/working')
files = {
    'remotion-curated.zip': f'{BASE}/data/remotion-curated.zip',
    'motion-ui-remotion-1000.zip': f'{BASE}/data/motion-ui-remotion-1000.zip',
    'train_qlora.py': f'{BASE}/training/train_qlora.py',
}
for name, url in files.items():
    target = WORK / name
    if not target.exists():
        print('Downloading', name)
        urlretrieve(url, target)
print('Files ready:', *(p.name for p in WORK.glob('*remotion*zip')))

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU accelerator in Notebook settings.')
print(torch.cuda.get_device_name(0))
print('GPU count:', torch.cuda.device_count())

In [ ]:
!python /kaggle/working/train_qlora.py \
  --curated-zip /kaggle/working/remotion-curated.zip \
  --motion-ui-zip /kaggle/working/motion-ui-remotion-1000.zip \
  --output-dir /kaggle/working/remotion-qlora \
  --max-length 4096 \
  --max-train-samples 1000 \
  --max-eval-samples 200

## Use the full training split

After the sample run fits your available runtime and memory, rerun the training cell with `--max-train-samples 0 --max-eval-samples 0`. This uses all 10,469 training rows and all 837 validation rows. It still never reads the 775-row test split.

Download `/kaggle/working/remotion-qlora` from the Notebook Output panel before the session expires. The saved files are adapter weights, not a standalone model; load them with the same base model and PEFT. See `training/README.md` for details and limitations.